In [ ]:
import os, sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"
os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import logging
import utils.logger as _log_mod
def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers: return logger
    logger.setLevel(logging.DEBUG)
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S"))
    logger.addHandler(h)
    return logger
_log_mod.setup_logger = setup_logger
print(f"CWD: {os.getcwd()}")

In [ ]:
!pip install lightgbm --quiet

In [ ]:
"""LightGBM Walk-Forward with Checkpoint Saving
각 윈도우별 LightGBM 모델을 joblib로 저장.
TFT WF와 동일한 윈도우/피처 사용.
"""
import warnings, json, joblib
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import accuracy_score, roc_auc_score

warnings.filterwarnings("ignore")
print(f"LightGBM: {lgb.__version__}")

In [ ]:
# ===== 설정 =====
BASE_PATH = Path(os.environ.get("BASE_PATH"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
MODEL_SAVE_PATH = BASE_PATH / "models" / "lgbm_wf"
CKPT_DIR = MODEL_SAVE_PATH / "checkpoints"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

WF_START = "2021-01-01"
WF_END = "2026-01-31"
WF_STEP_MONTHS = 3

CLEANED_FEATURES = ["vix_change", "vix", "macd_norm", "momentum_20d",
                    "relative_return", "high_low_range", "kospi200_return", "volume_ratio_5d"]

# 윈도우 생성 (TFT WF와 동일)
windows = []
current = pd.Timestamp(WF_START)
end = pd.Timestamp(WF_END)
while current < end:
    test_end = current + pd.DateOffset(months=WF_STEP_MONTHS) - pd.Timedelta(days=1)
    if test_end > end: test_end = end
    train_end = current - pd.Timedelta(days=1)
    windows.append((str(train_end.date()), str(current.date()), str(test_end.date())))
    current += pd.DateOffset(months=WF_STEP_MONTHS)

print(f"Walk-Forward 윈도우: {len(windows)}개")

In [ ]:
# ===== 데이터 로드 =====
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])
df["target_5d"] = df["target_5d"].astype(int)
CLEANED_FEATURES = [c for c in CLEANED_FEATURES if c in df.columns]
print(f"데이터: {len(df):,} rows, {len(CLEANED_FEATURES)} features")

In [ ]:
# ===== Walk-Forward 실행 =====
wf_results = []

for i, (train_end, test_start, test_end) in enumerate(windows):
    print(f"\n{'='*60}")
    print(f"  Window [{i:2d}/{len(windows)}] train_end={train_end}, test={test_start}~{test_end}")
    print(f"{'='*60}")
    
    ckpt_path = CKPT_DIR / f"window_{i:02d}_te_{train_end}.joblib"
    
    try:
        # 데이터 분할
        te_ts = pd.Timestamp(train_end)
        ts_ts = pd.Timestamp(test_start)
        tend_ts = pd.Timestamp(test_end)
        
        train_mask = df["date"] <= te_ts
        test_mask = (df["date"] >= ts_ts) & (df["date"] <= tend_ts)
        
        X_train = df[train_mask][CLEANED_FEATURES].values.astype(np.float32)
        y_train = df[train_mask]["target_5d"].values
        X_test = df[test_mask][CLEANED_FEATURES].values.astype(np.float32)
        y_test = df[test_mask]["target_5d"].values
        
        if len(X_train) < 1000 or len(X_test) < 100:
            print(f"  SKIP: train={len(X_train)}, test={len(X_test)}")
            wf_results.append({"window": i, "train_end": train_end, "test_start": test_start,
                               "test_end": test_end, "error": "데이터 부족"})
            continue
        
        # 체크포인트 로드 or 학습
        if ckpt_path.exists():
            print(f"  체크포인트 로드: {ckpt_path.name}")
            model = joblib.load(str(ckpt_path))
        else:
            print(f"  신규 학습... (train={len(X_train):,}, test={len(X_test):,})")
            
            # 검증용 분할 (마지막 10%)
            val_size = max(len(X_train) // 10, 500)
            X_tr, X_val = X_train[:-val_size], X_train[-val_size:]
            y_tr, y_val = y_train[:-val_size], y_train[-val_size:]
            
            model = lgb.LGBMClassifier(
                n_estimators=500, max_depth=6, learning_rate=0.05, num_leaves=31,
                subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
                class_weight="balanced", random_state=42, verbose=-1)
            
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                      callbacks=[lgb.early_stopping(50, verbose=False)])
            
            # 체크포인트 저장
            joblib.dump(model, str(ckpt_path))
            print(f"  체크포인트 저장: {ckpt_path.name}")
        
        # 예측 & 평가
        probs = model.predict_proba(X_test)[:, 1]
        preds = (probs >= 0.5).astype(int)
        da = accuracy_score(y_test, preds)
        try: auc = roc_auc_score(y_test, probs)
        except: auc = float("nan")
        
        result = {"window": i, "train_end": train_end, "test_start": test_start,
                  "test_end": test_end, "n_train": len(X_train), "n_test": len(X_test),
                  "da": round(da*100, 2), "auc": round(auc, 4),
                  "pos_rate": round(preds.mean()*100, 1), "ckpt": str(ckpt_path)}
        wf_results.append(result)
        print(f"  DA={da*100:.2f}%, AUC={auc:.4f}, 양성={preds.mean()*100:.1f}%")
        
    except Exception as e:
        print(f"  ERROR: {e}")
        wf_results.append({"window": i, "train_end": train_end, "test_start": test_start,
                           "test_end": test_end, "error": str(e)})

print(f"\nWalk-Forward 완료: {len(wf_results)}개 윈도우")
print(f"체크포인트: {CKPT_DIR}")

In [ ]:
# ===== 결과 요약 =====
valid = [r for r in wf_results if "error" not in r]
if valid:
    das = [r["da"] for r in valid]
    aucs = [r["auc"] for r in valid if not np.isnan(r["auc"])]
    print("="*70)
    print("  LightGBM Walk-Forward 결과")
    print("="*70)
    print(f"  윈도우: {len(valid)}개 / {len(wf_results)}개")
    print(f"  평균 DA: {np.mean(das):.2f}% (std={np.std(das):.2f})")
    print(f"  평균 AUC: {np.mean(aucs):.4f}")
    print(f"  DA 범위: {min(das):.2f}% ~ {max(das):.2f}%")
    print()
    for r in valid:
        print(f"  [{r['window']:2d}] {r['test_start']}~{r['test_end']}  DA={r['da']:6.2f}%  AUC={r['auc']:.4f}  양성={r['pos_rate']:5.1f}%")
    print("="*70)

In [ ]:
# ===== 저장 =====
json.dump(wf_results, open(str(MODEL_SAVE_PATH / f"wf_results_{datetime.now().strftime('%Y%m%d')}.json"), "w"),
          ensure_ascii=False, indent=2)
print(f"결과 저장: {MODEL_SAVE_PATH}")

## LightGBM Walk-Forward (체크포인트)\n\n- 3개월 단위 슬라이딩 윈도우 (TFT WF와 동일)\n- 각 윈도우 모델을 joblib로 저장\n- 재실행 시 체크포인트 로드 (수초)\n\n저장 위치: `models/lgbm_wf/checkpoints/window_XX_te_YYYY-MM-DD.joblib`